# Milestone 2


This milestone has been created to transition you from classical NLP techniques to modern, state-of-the-art deep learning architectures. It focuses on familiarizing you with the Hugging Face ecosystem, understanding how attention mechanisms create context-aware representations, and leveraging pre-trained models and zero-shot classification to drastically improve upon your baseline ranking metrics.

Suggested Readings:
- Hugging Face transformers & datasets Library Basics
- The Attention Mechanism and BERT/RoBERTa Architectures
- Dense Context-Aware Embeddings (e.g., Sentence-Transformers)
- Zero-Shot Classification
- Softmax vs. Independent Sigmoid (Multi-label) Probabilities
- Prompting Small Language Models (SLMs) for Generative QA

## Introduction to Hugging Face transformers and datasets

Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [1]:
from datasets import load_dataset

train_dataset = load_dataset("csv", data_files= "../../data/raw/train.csv")['train']

In [2]:
train_dataset=train_dataset.map(lambda x: {**x, "combined_text": x["prompt"] + " " + x["A"]})

In [3]:
len(train_dataset[51]['combined_text'])

614

Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [4]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [9]:
tokenizer.vocab_size

30522

Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [10]:
print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [11]:
prompts=list(train_dataset["prompt"])
encoding=tokenizer(prompts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
encoding['input_ids'].shape

torch.Size([2000, 128])

## BERT/RoBERTa Architecture & Attention Mechanisms

A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?

In [15]:
print(768/12)

64.0


Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

Note: We follow zero-indexing here.

In [12]:
from transformers import AutoModel
model=AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
sample=tokenizer(train_dataset["prompt"][0], return_tensors="pt")
output=model(**sample)
output.last_hidden_state.shape

torch.Size([1, 31, 768])

Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).

In [14]:
round(output.last_hidden_state[0, 0, :5].sum().item(), 4)

-1.2001

Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [15]:
model2=AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
encoding=tokenizer("Light-ion fusion is a technique.", return_tensors="pt")

tokens=tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])
print(tokens)
tokens.index('fusion')


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


4

In [17]:
output=model2(**encoding)

attention_last_layer=output.attentions[-1]
print(attention_last_layer.shape)
first_attention_head=attention_last_layer[0, 0, :, :]
round(first_attention_head[0, tokens.index('fusion')].item(), 4)

torch.Size([1, 12, 10, 10])


0.1025

## Context-Aware Embeddings

Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [18]:
from sentence_transformers import SentenceTransformer, util

st_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
prompt=train_dataset["prompt"][0]
option_b=train_dataset["B"][0]

embeddings=st_model.encode([prompt, option_b], convert_to_tensor=True)

In [20]:
round(util.cos_sim(embeddings[0], embeddings[1]).item(), 4)

0.7658

Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?

In [ ]:
# Pipeline 1 - TF-IDF Vectorization
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df=pd.DataFrame(train_dataset)
labels=["A", "B", "C", "D", "E"]

vectorizer=TfidfVectorizer()
tfidf_preds=[]

for _, row in df.iterrows():
    texts=[row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]
    X=vectorizer.fit_transform(texts)

    # similarity bw prompt and options
    cosine_similarities=cosine_similarity(X[0], X[1:])[0]

    ordered_indices=cosine_similarities.argsort()[::-1]

    pred=[labels[i] for i in ordered_indices[:3]]
    tfidf_preds.append(pred)

In [52]:
# Pipeline 2 - Sentence Transformers Embeddings
model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

minilm_preds=[]

for _, row in df.iterrows():
    texts=[row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]

    embeddings=model.encode(texts, convert_to_tensor=True)

    cosine_similarities=util.cos_sim(embeddings[0], embeddings[1:])[0]

    ordered_indices=cosine_similarities.argsort(descending=True)

    pred=[labels[i] for i in ordered_indices[:3]]
    minilm_preds.append(pred)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
answers=list(df['answer'])

def average_prediction_score3(prediction:list, ground_truth:str) -> float:
    if not prediction:
        return 0.0
    score = 0.0
    for i, pred in enumerate(prediction[:3]):
        if pred == ground_truth:
            score += 1 / (i + 1)
    return score

def mean_average_precision3(predictions:list, ground_truths:list) -> float:
    if not predictions or not ground_truths:
        return 0.0
    total_score = 0.0
    for pred, gt in zip(predictions, ground_truths):
        total_score += average_prediction_score3(pred, gt)
    return total_score / len(predictions)


print("TF-IDF MAP@3:", mean_average_precision3(tfidf_preds, answers))
print("MiniLM MAP@3:", mean_average_precision3(minilm_preds, answers))

TF-IDF MAP@3: 0.3269166666666673
MiniLM MAP@3: 0.42308333333333487


In [54]:
# improved count
improved_count = 0
for tfidf_pred,minilm_pred,ans in zip(tfidf_preds, minilm_preds,answers):
    if (ans not in tfidf_pred) and (ans in minilm_pred):
        improved_count += 1
print(f"Number of improved predictions: {improved_count}")

Number of improved predictions: 564


## Zero-shot classification concepts

Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

row=train_dataset[1]

candidates=[row['A'], row['B'], row['C']]

result = classifier(row['prompt'], candidates)
print(result)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

c:\Users\dhruv\miniconda3\envs\mcq_solver\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dhruv\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [7]:
print(round(result['scores'][0], 4))

0.4575


Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [8]:
result2=classifier(row['prompt'], candidates, multi_label=True)
print(result2)

print(abs(sum(result['scores'])-sum(result2['scores'])))

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

Let's try Generative AI instead of Classification.

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B."

Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?

In [20]:
# # Error due to vesion issue. text2text-generation is not available
# generator=pipeline("text2text-generation", model="google/flan-t5-small")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row=train_dataset[0]

prompt=(f"Question: {row['prompt']}. Is the correct answer A: {row['A']} or B: {row['B']}? Answer with just the letter A or B.")

inputs=tokenizer(prompt, return_tensors="pt")

output = model.generate(**inputs, max_new_tokens=5)
answer = tokenizer.decode(output[0], skip_special_tokens=True)

print("Answer: ", answer)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Answer:  B


In [12]:
import transformers
print(transformers.__version__)

5.3.0
